# P1 ML Core — Voice Cloning Detection (Colab)
This notebook is your **P1 (ML Core)** workspace. Run top to bottom.

**What you own:** audio preprocessing, deepfake-audio model inference, voiceprint (speaker embedding), prosody features, and the accuracy numbers (English + Hindi).

**Flow:** install deps → test detection → enroll a voice → test voiceprint → extract prosody → measure accuracy on real vs cloned clips.


In [ ]:
# @title 1. Mount Drive + clone repo + hydrate dataset (folium-style)
from google.colab import drive
from pathlib import Path
import sys, os, pathlib, subprocess

drive.mount("/content/drive")

REPO_URL = "https://github.com/io-PEAK/VoxDetect.git"   # change if forked
REPO_DIR = Path("/content/VoxDetect")                    # session-only git clone

# ---- Durable on Google Drive (persist across sessions) ----
ML_BASE        = Path("/content/drive/MyDrive/VoxDetect/ml-core")
DATASET_DIR    = ML_BASE / "dataset"        # test_data.zip + .manifest.json (upload once)
RESULTS_DIR    = ML_BASE / "results"        # every evaluate run writes a UNIQUE <sprint>.json
CHECKPOINT_DIR = ML_BASE / "checkpoints"     # model artifacts
results_dir = RESULTS_DIR
for d in (DATASET_DIR, RESULTS_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Session-local (rebuilt each run from the Drive archive) ----
LOCAL_DATA_DIR = Path("/content/VoxDetect_data")   # hydrated test_data (real/ + cloned/)

# ---- git clone the repo (session-only) so we get latest src/ + scripts/ ----
if not (REPO_DIR / "ml-core" / "src").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

SRC_PKG = REPO_DIR / "ml-core" / "src"
sys.path.insert(0, str(SRC_PKG))

# ---- hydrate local test_data from the Drive archive (mirrors folium organize_datasets) ----
subprocess.run([
    sys.executable, str(REPO_DIR / "ml-core" / "scripts" / "organize_dataset.py"),
    "--raw-dir", str(DATASET_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
])
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
test_data_dir = LOCAL_DATA_DIR   # evaluate.py reads real/ + cloned/ from here

print("src/ package at:", SRC_PKG)
print("DATASET_DIR (Drive):", DATASET_DIR)
print("LOCAL_DATA_DIR   :", LOCAL_DATA_DIR)
print("RESULTS_DIR (Drive):", RESULTS_DIR)
print("CHECKPOINT_DIR  :", CHECKPOINT_DIR)

# Install audio + ML deps (no stray 'audio' package)
!pip install -q torch torchaudio librosa soundfile transformers resemblyzer huggingface_hub numpy scipy
print("deps installed")

In [ ]:
# @title 2. Load audio + preprocess (sanity check)
from audio_utils import load_audio, preprocess_clip, chunk_audio
import numpy as np, glob, pathlib

# test_data_dir = /content/VoxDetect_data, hydrated from the DRIVE archive by cell 1.
# Pick a REAL clip from the hydrated dataset (real/english/<speaker>/*.wav).
real_clips = sorted(glob.glob(str(test_data_dir / "real/**/*.wav"), recursive=True))
if not real_clips:
    raise SystemExit(
        "[FAIL] No real clips hydrated under %s/real. The Drive archive (test_data.zip + "
        "test_data.zip.manifest.json) must sit in MyDrive/VoxDetect/ml-core/dataset and "
        "hydrate 48 clips. Check cell 1 output, then re-run." % test_data_dir
    )
sample_path = real_clips[0]
wav, sr = load_audio(sample_path)
print("loaded REAL clip:", pathlib.Path(sample_path).name, wav.shape, "at", sr, "Hz")


In [ ]:
# @title 3. Deepfake detection with pretrained model
# Gustking/wav2vec2-large-xlsr-deepfake-audio-classification (deepfake audio).
from detect import DetectionEngine, quick_check

# This may ask for an HF token if the model is gated. Set HF_TOKEN in secrets.
eng = DetectionEngine(model_variant="wav2vec2")

# Score the SAME real clip loaded in cell 2 -> expect LOW risk
try:
    r = eng.analyze_audio(sample_path)          # sample_path set in cell 2
    print("real clip:", pathlib.Path(sample_path).name, "risk_score:", r["risk_score"], r["band"], r["signals"])
except Exception as e:
    print("Detection test failed:", e)


In [ ]:
# @title 4. FIRST-RUN VALIDATION — do this before trusting any numbers
# Confirms the model's label mapping AND checks risk scores on a real + cloned clip.
# real  clip should score LOW (< threshold)
# cloned clip should score HIGH (>= threshold)
import glob
import validate

real_clip = sorted(glob.glob(str(test_data_dir / "real/**/*.wav"), recursive=True))[0]
cloned_clip = sorted(glob.glob(str(test_data_dir / "cloned/**/*.wav"), recursive=True))[0]
print("real clip  :", real_clip)
print("cloned clip:", cloned_clip)

validate.run(real=real_clip, cloned=cloned_clip, threshold=70)
print("\nPASS if real scored LOW and cloned scored HIGH. If reversed, the fake class "
      "index is swapped -> fix `_find_fake_index` in detect.py.")


In [ ]:
# @title 5. Voiceprint: enroll + compare
# Enroll one real clip, then compare against another clip from the SAME speaker.
# Same-speaker similarity should be high (cosine ~0.5-0.9). A cloned clip of that
# speaker should be LOWER -> that gap is the voiceprint signal.
import glob
from voiceprint import Voiceprint

real_clips = sorted(glob.glob(str(test_data_dir / "real/**/*.wav"), recursive=True))
vp = Voiceprint()
try:
    enrolled = vp.enroll(real_clips[0], label="speaker")
    print("enrolled embedding dim:", len(enrolled["embedding"]))
    emb2 = vp.embed(real_clips[1])                      # another real clip (same dataset)
    sim_same = vp.similarity(enrolled["embedding"], emb2)
    print("real-vs-real similarity (higher = better): %.3f" % sim_same)

    cloned_clips = sorted(glob.glob(str(test_data_dir / "cloned/**/*.wav"), recursive=True))
    emb3 = vp.embed(cloned_clips[0])
    sim_clone = vp.similarity(enrolled["embedding"], emb3)
    print("real-vs-cloned similarity (lower = more suspicious): %.3f" % sim_clone)
except Exception as e:
    print("Voiceprint error (needs at least 2 real clips):", e)


In [ ]:
# @title 6. Prosody features
# Extract pitch variance / pause ratio / speaking rate from a real clip and compute
# a prosody anomaly score (higher = more synthetic by the generic natural-speech rule).
from prosody import extract_prosody, prosody_anomaly_score

try:
    feat = extract_prosody(sample_path)                 # sample_path set in cell 2
    print("prosody:", feat)
    print("anomaly:", round(prosody_anomaly_score(feat), 3))
except Exception as e:
    print("Prosody needs a real clip. Error:", e)


In [ ]:
# @title 7. (Coming) Build your own real vs cloned dataset
# P4 provides real/ + cloned/ clips -> upload once to Drive (scripts/upload_dataset.py).
# Every session hydrates them locally, then sprint1/sprint2 run evaluate.py over them
# and write results/<sprint>.json back to Drive (RESULTS_DIR).
print("Dataset build + measurement: run sprint1/sprint2 notebooks (results go to RESULTS_DIR on Drive).")
